# 2단계: 사전학습 모델의 인스트럭션 추가 학습 (think weight 0.1)

먼저 `colab_pretrain.ipynb`를 실행해 Drive에 `korean_sllm_data/pretrain/best.pt`를 만드세요.
이 노트북은 해당 체크포인트의 **모델 구조와 가중치만** 불러와 SFT를 시작합니다.
옵티마이저, 학습 스텝, best loss는 초기화하며 SFT 결과는 `korean_sllm_data/pair`에 저장합니다.
토크나이저는 사전학습 폴더의 `spm.model`을 복사하고 SHA-256 일치 여부를 검사합니다.
모델은 기존 Gemma 계열 24L/d768 + MTP 16 구조를 유지합니다.

- 데이터: `korean_sllm_data/pair/train.jsonl`, `val.jsonl`의 `{"user": "질문", "assistant": "답변"}` 형식.
  아래 셀에서 Drive 경로를 지정할 수 있습니다. 기존 저장소의 tar.xz도 지원합니다.
- 기본 설정은 기존 **RTX PRO 6000 96GB** 기준(seq 2048, batch 8, accum 4)입니다.
  소형 GPU에서는 batch/seq를 낮추고 gradient checkpointing을 켜세요. 전체 구조의 메모리 적합성은 실측이 필요합니다.
- SFT learning rate는 시작값 `5e-5`, epochs는 2로 설정했습니다. 검증 loss와 실제 답변 품질을 보며 조정하세요.
- user/prompt weight=0, `<think>`/`</think>` 태그=1, reasoning 내부=0.1, 최종 답변/종료=1.
- 재개 시 2번 셀의 `RESUME = str(Path(CKPT_DIR) / 'last.pt')`로 설정하세요. `pair/spm.model`을 복원합니다.
- 신규 SFT는 `--init-from`, 중단된 SFT의 재개는 `--resume`을 사용합니다. 두 옵션은 동시에 사용할 수 없습니다.
- `best.pt`는 검증 main loss 최저, `last.pt`는 재개용입니다. 사전학습과 SFT loss는 목적이 달라 직접 비교하지 않습니다.
- 일반 텍스트 사전학습이 품질 개선을 보장하지는 않습니다. SFT-only 모델과 동일한 미학습 평가셋에서 비교하세요.


In [ ]:
# 1) 리포 준비 (Colab GPU 런타임을 먼저 선택하세요)
from pathlib import Path
import subprocess
import os
REPO_URL = "https://github.com/MinsuChae/korean_sllm.git"
REPO_DIR = Path('/content/korean_sllm')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt', 'protobuf>=4.25.0'], check=True)


In [ ]:
# 이 노트북에 포함된 학습 코드를 사용합니다 (GitHub 반영 전에도 실행 가능).
from pathlib import Path
Path('train.py').write_text('"""학습 스크립트 (Colab / 로컬 공용).\n\n  python train.py                          # base 프리셋 (24L, d768), GPU 권장\n  python train.py --preset tiny --max-steps 30   # 로컬 smoke test\n\n- {train,val}.jsonl 이 없으면 tar.xz 에서 자동으로 푼다 (data.py).\n- bf16 지원 GPU 는 bf16 autocast, 그 외 CUDA 는 fp16+GradScaler, CPU 는 fp32.\n- --ckpt-dir 에 last.pt(--save-every 주기 최신)와 best.pt(val main_loss 최저)만 유지하고,\n  --resume 은 last.pt 로 재개한다 (Google Drive 경로 가능).\n"""\n\nimport argparse\nimport hashlib\nimport math\nimport os\nimport time\nfrom pathlib import Path\n\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom data import load_tokenizer, make_dataset\nfrom model import KoreanSLLM, ModelConfig\n\nROOT = Path(__file__).resolve().parent\n\nPRESETS = {\n    "base": {},\n    "tiny": {"n_layers": 2, "d_model": 64, "n_heads": 4, "n_kv_heads": 2, "head_dim": 16,\n             "ffn_hidden": 128, "mtp_ffn_hidden": 64, "max_seq_len": 128, "sliding_window": 32},\n}\n\n\ndef parse_args():\n    p = argparse.ArgumentParser(description="Korean sLLM 학습")\n    p.add_argument("--preset", choices=PRESETS, default="base")\n    p.add_argument("--data-dir", default=str(ROOT))\n    p.add_argument("--cache-dir", default=str(ROOT / "cache"))\n    p.add_argument("--ckpt-dir", default=str(ROOT / "checkpoints"))\n    p.add_argument("--stage", choices=("sft", "pretrain"), default="sft")\n    checkpoint = p.add_mutually_exclusive_group()\n    checkpoint.add_argument("--resume", default=None, help="동일 단계의 모델/옵티마이저/스텝 복원")\n    checkpoint.add_argument("--init-from", default=None, help="사전학습 모델 가중치만 복원; 새 스케줄 시작")\n    p.add_argument("--seq-len", type=int, default=None, help="기본: config.max_seq_len")\n    p.add_argument("--max-sample-len", type=int, default=None,\n                   help="이 토큰 길이를 넘는 샘플은 캐시에서 제외. 기본: seq_len 과 동일, 0 이면 제외하지 않음")\n    p.add_argument("--batch-size", type=int, default=8)\n    p.add_argument("--grad-accum", type=int, default=4)\n    p.add_argument("--max-steps", type=int, default=20_000)\n    p.add_argument("--epochs", type=float, default=None,\n                   help="지정 시 max-steps 를 무시하고 데이터셋 윈도우 수에서 스텝을 환산 (권장: 4)")\n    p.add_argument("--lr", type=float, default=3e-4)\n    p.add_argument("--min-lr-ratio", type=float, default=0.1)\n    p.add_argument("--warmup-steps", type=int, default=500)\n    p.add_argument("--weight-decay", type=float, default=0.1)\n    p.add_argument("--grad-clip", type=float, default=1.0)\n    p.add_argument("--eval-every", type=int, default=500)\n    p.add_argument("--eval-batches", type=int, default=50)\n    p.add_argument("--save-every", type=int, default=1000)\n    p.add_argument("--log-every", type=int, default=20)\n    p.add_argument("--grad-checkpointing", action="store_true")\n    p.add_argument("--num-workers", type=int, default=2)\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--sample", default=None, help="학습 종료 후 이 프롬프트로 생성 데모")\n    return p.parse_args()\n\n\ndef lr_at(step: int, args) -> float:\n    if step < args.warmup_steps:\n        return args.lr * (step + 1) / args.warmup_steps\n    t = (step - args.warmup_steps) / max(args.max_steps - args.warmup_steps, 1)\n    return args.lr * (args.min_lr_ratio + (1 - args.min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * min(t, 1.0))))\n\n\ndef setup_amp(device: torch.device):\n    if device.type == "cuda" and torch.cuda.is_bf16_supported():\n        return torch.bfloat16, None\n    if device.type == "cuda":\n        return torch.float16, torch.amp.GradScaler("cuda")\n    return None, None  # CPU: fp32\n\n\n@torch.no_grad()\ndef evaluate(model, loader, device, amp_dtype, max_batches: int) -> dict[str, float]:\n    model.eval()\n    sums = {"main_loss": 0.0, "mtp_loss": 0.0}\n    n = 0\n    for batch in loader:\n        if n >= max_batches:\n            break\n        ids = batch["input_ids"].to(device)\n        mask = batch["loss_mask"].to(device)\n        with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None):\n            out = model(ids, mask)\n        sums["main_loss"] += out["main_loss"].item()\n        sums["mtp_loss"] += out["mtp_loss"].item()\n        n += 1\n    model.train()\n    return {k: v / max(n, 1) for k, v in sums.items()}\n\n\ndef save_ckpt(path: Path, model, optim, step: int, cfg: ModelConfig, best_val: float,\n              stage: str, tokenizer_sha256: str, scaler=None):\n    # 임시 파일에 쓴 뒤 교체 - Drive 위에서 덮어쓰기 도중 런타임이 끊겨도 기존 파일이 보존된다\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(".tmp")\n    torch.save({"model": model.state_dict(), "optim": optim.state_dict(),\n                "step": step, "config": cfg.to_dict(), "best_val": best_val,\n                "stage": stage, "tokenizer_sha256": tokenizer_sha256,\n                "scaler": scaler.state_dict() if scaler else None}, tmp)\n    os.replace(tmp, path)\n    print(f"[ckpt] step {step} -> {path}")\n\n\ndef main():\n    args = parse_args()\n    torch.manual_seed(args.seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n    if args.init_from and args.stage != "sft":\n        raise ValueError("--init-from은 pretrain -> sft 전환에 사용하세요.")\n    ckpt = None\n    checkpoint_path = args.resume or args.init_from\n    if checkpoint_path:\n        ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n        expected_stage = "pretrain" if args.init_from else args.stage\n        if ckpt.get("stage", "sft") != expected_stage:\n            raise ValueError("체크포인트 단계 불일치: pretrain -> sft에는 --init-from을 사용하세요.")\n        if args.init_from and Path(args.ckpt_dir).resolve() == Path(args.init_from).resolve().parent:\n            raise ValueError("SFT 출력 폴더는 사전학습 체크포인트 폴더와 분리하세요.")\n    cfg = ModelConfig(**ckpt["config"]) if ckpt else ModelConfig(**PRESETS[args.preset])\n    seq_len = args.seq_len or cfg.max_seq_len\n    if not 2 <= seq_len <= cfg.max_seq_len:\n        raise ValueError(f"seq-len은 2..{cfg.max_seq_len} 범위여야 합니다.")\n    if min(args.batch_size, args.grad_accum, args.eval_every, args.save_every,\n           args.eval_batches, args.log_every, args.max_steps) < 1:\n        raise ValueError("배치/스텝/주기 값은 양수여야 합니다.")\n    if args.epochs is not None and args.epochs <= 0:\n        raise ValueError("epochs는 양수여야 합니다.")\n    root, cache_dir, ckpt_dir = Path(args.data_dir), Path(args.cache_dir), Path(args.ckpt_dir)\n\n    sp = load_tokenizer()\n    tokenizer_sha256 = hashlib.sha256(sp.serialized_model_proto()).hexdigest()\n    if ckpt:\n        saved_hash = ckpt.get("tokenizer_sha256")\n        if args.init_from and not saved_hash:\n            raise ValueError("토크나이저 해시가 있는 새 pretrain 체크포인트가 필요합니다.")\n        if saved_hash and saved_hash != tokenizer_sha256:\n            raise ValueError("토크나이저가 체크포인트와 다릅니다. 사전학습 때의 spm.model을 사용하세요.")\n    assert sp.get_piece_size() == cfg.vocab_size, \\\n        f"토크나이저 vocab {sp.get_piece_size()} != config {cfg.vocab_size}"\n\n    # seq_len 보다 긴 샘플은 어차피 한 윈도우에 못 들어가므로 기본으로 제외한다 (--max-sample-len 0 으로 해제)\n    max_sample_len = seq_len if args.max_sample_len is None else (args.max_sample_len or None)\n    if args.stage == "pretrain":\n        from pretrain_data import make_pretrain_dataset\n        train_ds = make_pretrain_dataset(root, "train", seq_len, cache_dir, sp)\n        val_ds = make_pretrain_dataset(root, "val", seq_len, cache_dir, sp)\n    else:\n        train_ds = make_dataset(root, "train", seq_len, cache_dir, sp, max_sample_len)\n        val_ds = make_dataset(root, "val", seq_len, cache_dir, sp, max_sample_len)\n    if len(train_ds) < args.batch_size or len(val_ds) == 0:\n        raise ValueError("학습 데이터가 한 배치 미만이거나 검증 데이터가 비었습니다. 데이터/seq-len/batch-size를 확인하세요.")\n    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,\n                              num_workers=args.num_workers, drop_last=True, pin_memory=device.type == "cuda")\n    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,\n                            num_workers=args.num_workers, drop_last=False)\n\n    # 토크나이저·seq_len 이 바뀌면 1 epoch 스텝 수가 달라지므로 epoch 로 지정할 수 있게 한다\n    steps_per_epoch = max(len(train_ds) // (args.batch_size * args.grad_accum), 1)\n    if args.epochs is not None:\n        args.max_steps = max(round(args.epochs * steps_per_epoch), 1)\n    tokens_per_step = args.batch_size * args.grad_accum * seq_len\n\n    model = KoreanSLLM(cfg).to(device)\n    print(f"모델: {cfg.n_layers}L d{cfg.d_model} v{cfg.vocab_size} | 파라미터 {model.num_params() / 1e6:.1f}M | "\n          f"MTP {cfg.mtp_n}토큰 | device={device} | train {len(train_ds):,} windows(seq {seq_len})")\n    print(f"스케줄: {args.max_steps:,} steps × {tokens_per_step:,} tok/step "\n          f"= {args.max_steps * tokens_per_step / 1e9:.2f}B tokens ≈ {args.max_steps / steps_per_epoch:.1f} epochs")\n\n    if args.grad_checkpointing:\n        import functools\n        from torch.utils.checkpoint import checkpoint as checkpoint_forward\n        for block in model.blocks:\n            block._orig_forward = block.forward\n            block.forward = functools.partial(\n                checkpoint_forward, block._orig_forward, use_reentrant=False)\n\n    decay, no_decay = [], []\n    for name, param in model.named_parameters():\n        (no_decay if param.ndim < 2 else decay).append(param)\n    optim = torch.optim.AdamW(\n        [{"params": decay, "weight_decay": args.weight_decay},\n         {"params": no_decay, "weight_decay": 0.0}],\n        lr=args.lr, betas=(0.9, 0.95), fused=device.type == "cuda")\n\n    amp_dtype, scaler = setup_amp(device)\n    print(f"정밀도: {amp_dtype or torch.float32}")\n\n    step = 0\n    best_val = float("inf")\n    if ckpt:\n        model.load_state_dict(ckpt["model"])\n    if args.init_from:\n        print(f"[init] {args.init_from}: 모델 구조/가중치 복원, optimizer/step/best_val 초기화")\n    if args.resume:\n        optim.load_state_dict(ckpt["optim"])\n        if scaler and ckpt.get("scaler"):\n            scaler.load_state_dict(ckpt["scaler"])\n        step = ckpt["step"]\n        best_val = ckpt.get("best_val", float("inf"))  # 미복원 시 첫 eval 이 best.pt 를 덮어쓴다\n        print(f"[ckpt] {args.resume} 에서 step {step} 재개 (best val {best_val:.4f})")\n    del ckpt\n\n    model.train()\n    data_iter = iter(train_loader)\n    t0, tokens_seen = time.time(), 0\n    while step < args.max_steps:\n        for g in optim.param_groups:\n            g["lr"] = lr_at(step, args)\n        optim.zero_grad(set_to_none=True)\n        logs = {"loss": 0.0, "main_loss": 0.0, "mtp_loss": 0.0}\n        for _ in range(args.grad_accum):\n            try:\n                batch = next(data_iter)\n            except StopIteration:\n                data_iter = iter(train_loader)\n                batch = next(data_iter)\n            ids = batch["input_ids"].to(device)\n            mask = batch["loss_mask"].to(device)\n            with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None):\n                out = model(ids, mask)\n            loss = out["loss"] / args.grad_accum\n            (scaler.scale(loss) if scaler else loss).backward()\n            for k in logs:\n                logs[k] += out[k].item() / args.grad_accum\n            tokens_seen += ids.numel()\n\n        if scaler:\n            scaler.unscale_(optim)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)\n        if scaler:\n            scaler.step(optim)\n            scaler.update()\n        else:\n            optim.step()\n        step += 1\n\n        if step % args.log_every == 0:\n            tps = tokens_seen / (time.time() - t0)\n            print(f"step {step:6d} | loss {logs[\'loss\']:.4f} (main {logs[\'main_loss\']:.4f} "\n                  f"mtp {logs[\'mtp_loss\']:.4f}) | lr {optim.param_groups[0][\'lr\']:.2e} | {tps / 1e3:.1f}k tok/s")\n        if step % args.eval_every == 0 or step == args.max_steps:\n            ev = evaluate(model, val_loader, device, amp_dtype, args.eval_batches)\n            mem = (f" | mem {torch.cuda.max_memory_allocated() / 2**30:.1f}GiB"\n                   if device.type == "cuda" else "")\n            print(f"  [val] main {ev[\'main_loss\']:.4f} | mtp {ev[\'mtp_loss\']:.4f} | "\n                  f"ppl {math.exp(min(ev[\'main_loss\'], 20)):.1f}{mem}")\n            if ev["main_loss"] < best_val:\n                best_val = ev["main_loss"]\n                print(f"  [ckpt] new best (val main {best_val:.4f})")\n                save_ckpt(ckpt_dir / "best.pt", model, optim, step, cfg, best_val,\n                          args.stage, tokenizer_sha256, scaler)\n        if step % args.save_every == 0 or step == args.max_steps:\n            save_ckpt(ckpt_dir / "last.pt", model, optim, step, cfg, best_val,\n                      args.stage, tokenizer_sha256, scaler)\n\n    print(f"학습 종료: last.pt step {step} | best.pt val main {best_val:.4f}")\n\n    if args.sample:\n        from data import encode_sample\n        prompt_ids = encode_sample(sp, args.sample, "")[0][:-2]  # assistant 답변/eos 제외\n        out = model.generate(torch.tensor([prompt_ids], device=device), max_new_tokens=256)\n        print("\\n=== 생성 데모 ===")\n        print(sp.decode(out[0].tolist()))\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
Path('pretrain_data.py').write_text('"""Plain-text JSONL -> disk-backed causal-LM windows (no chat template)."""\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset\n\n\nclass PretrainDataset(Dataset):\n    def __init__(self, path: Path, seq_len: int):\n        self.tokens = np.memmap(path, dtype=np.uint16, mode="r")\n        self.seq_len = seq_len\n\n    def __len__(self):\n        # One extra target token; adjacent windows overlap by exactly one token.\n        return max(0, (len(self.tokens) - 1) // (self.seq_len - 1))\n\n    def __getitem__(self, index):\n        start = index * (self.seq_len - 1)\n        ids = torch.from_numpy(self.tokens[start:start + self.seq_len].astype(np.int64))\n        return {"input_ids": ids, "loss_mask": torch.ones_like(ids)}\n\n\ndef make_pretrain_dataset(root, split, seq_len, cache_dir, sp):\n    source = Path(root) / f"{split}.jsonl"\n    if not source.is_file():\n        raise FileNotFoundError(f\'{source}: {{"text": "본문"}} 형식의 JSONL을 준비하세요.\')\n    if not 0 <= sp.bos_id() < sp.get_piece_size() or not 0 <= sp.eos_id() < sp.get_piece_size():\n        raise ValueError("BOS/EOS가 있는 기존 토크나이저가 필요합니다.")\n    if sp.get_piece_size() > 65536:\n        raise ValueError("uint16 vocabulary overflow")\n    digest = hashlib.sha256(sp.serialized_model_proto())\n    with source.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    cache_dir = Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    target = cache_dir / f"pretrain_v1_{split}_{digest.hexdigest()}.bin"\n    if not target.exists():\n        tmp = target.with_suffix(".tmp")\n        documents = tokens = 0\n        try:\n            with source.open(encoding="utf-8") as stream, tmp.open("wb") as out:\n                for line_no, line in enumerate(stream, 1):\n                    if not line.strip():\n                        continue\n                    try:\n                        obj = json.loads(line)\n                    except json.JSONDecodeError as exc:\n                        raise ValueError(f"{source}:{line_no}: 잘못된 JSON") from exc\n                    text = obj.get("text") if isinstance(obj, dict) else None\n                    if not isinstance(text, str):\n                        raise ValueError(f"{source}:{line_no}: text 문자열이 필요합니다.")\n                    if not text.strip():\n                        continue\n                    ids = [sp.bos_id()] + sp.encode(text) + [sp.eos_id()]\n                    np.asarray(ids, dtype=np.uint16).tofile(out)\n                    documents += 1\n                    tokens += len(ids)\n            if tokens == 0:\n                raise ValueError(f"{source}: 비어 있는 코퍼스")\n            os.replace(tmp, target)\n        finally:\n            tmp.unlink(missing_ok=True)\n        print(f"[pretrain] {split}: {documents:,} documents / {tokens:,} tokens")\n    dataset = PretrainDataset(target, seq_len)\n    if len(dataset) == 0:\n        raise ValueError(f"{source}: 최소 {seq_len} tokens가 필요합니다.")\n    return dataset\n', encoding="utf-8")
print("학습 코드 준비 완료")


In [ ]:
# 1b) <think> token loss weighting 패치
# user/prompt=0.0, <think>/</think>=1.0, think 내부=0.1, 최종 답변/종료=1.0
from pathlib import Path

data_py = Path("data.py")
src = data_py.read_text(encoding="utf-8")

func_start = src.find("def encode_sample(")
func_end = src.find("\ndef build_cache(", func_start)
if func_start < 0 or func_end < 0:
    raise RuntimeError("data.py의 encode_sample/build_cache 위치를 찾지 못했습니다.")

new_encode = '''THINK_OPEN_TAG = "<think>"
THINK_END_TAG = "</think>"
THINK_INNER_WEIGHT = 0.1


def encode_sample(sp: spm.SentencePieceProcessor, user: str, assistant: str) -> tuple[list[int], list[float]]:
    prompt_ids = [sp.bos_id()] + sp.encode(f"<start_of_turn>user\\n{user}<end_of_turn>\\n<start_of_turn>model\\n")

    # 전체 assistant를 한 번에 tokenize: 기존 tokenization 보존
    answer_text = f"{assistant}<end_of_turn>"
    proto = sp.encode(answer_text, out_type="proto")
    answer_ids = [piece.id for piece in proto.pieces] + [sp.eos_id()]

    # assistant 기본값은 1.0: 태그, 최종 답변, end_of_turn, EOS
    answer_weights = [1.0] * len(proto.pieces) + [1.0]

    # SentencePiece piece.begin/end는 byte offset.
    # well-formed <think>...</think>의 태그 "사이"에 완전히 포함된 piece만 0.1.
    # 태그 경계를 걸치는 piece는 1.0을 유지한다.
    assistant_bytes = assistant.encode("utf-8")
    open_tag = THINK_OPEN_TAG.encode("utf-8")
    close_tag = THINK_END_TAG.encode("utf-8")
    search_pos = 0

    while True:
        open_start = assistant_bytes.find(open_tag, search_pos)
        if open_start < 0:
            break

        inner_start = open_start + len(open_tag)
        close_start = assistant_bytes.find(close_tag, inner_start)
        if close_start < 0:
            # malformed/unclosed <think>는 최종 답변까지 잘못 0.1로 만들지 않도록 1.0 유지
            break

        for i, piece in enumerate(proto.pieces):
            if piece.begin >= inner_start and piece.end <= close_start:
                answer_weights[i] = THINK_INNER_WEIGHT

        search_pos = close_start + len(close_tag)

    ids = prompt_ids + answer_ids
    weights = [0.0] * len(prompt_ids) + answer_weights
    return ids, weights
'''

src = src[:func_start] + new_encode + src[func_end + 1:]

# 기존 mask cache와 분리
lines = src.splitlines()
for i, line in enumerate(lines):
    if line.strip().startswith('suffix = f"_v{sp.get_piece_size()}_{sp_hash}'):
        indent = line[:len(line) - len(line.lstrip())]
        lines[i] = indent + 'suffix = f"_v{sp.get_piece_size()}_{sp_hash}_thinkw010_v1" + (f"_max{max_sample_len}" if max_sample_len else "")'
        break
else:
    raise RuntimeError("data.py의 cache suffix 줄을 찾지 못했습니다.")
src = "\n".join(lines) + ("\n" if src.endswith("\n") else "")

# 0.1 저장/전달 가능하도록 float 사용
src = src.replace("masks: list[int] = []", "masks: list[float] = []")
src = src.replace(
    "np.save(mask_path, np.asarray(masks, dtype=np.uint8))",
    "np.save(mask_path, np.asarray(masks, dtype=np.float16))",
)
src = src.replace(
    '"loss_mask": torch.from_numpy(self.mask[s:e].astype(np.int64)),',
    '"loss_mask": torch.from_numpy(self.mask[s:e].astype(np.float32)),',
)

src = src.replace(
    "- 전처리 1회: jsonl 스트리밍 토크나이즈 -> 평탄한 uint16 토큰 배열 + uint8 손실 마스크를",
    "- 전처리 1회: jsonl 스트리밍 토크나이즈 -> 평탄한 uint16 토큰 배열 + float16 손실 가중치를",
)
src = src.replace(
    "- 손실 마스크: assistant 응답(+종료 토큰) 구간만 1, user/템플릿 구간은 0.",
    "- 손실 가중치: user/템플릿=0, <think>/</think>=1, think 내부=0.1, 최종 답변(+종료 토큰)=1.",
)
src = src.replace(
    "- 손실 마스크: assistant에 </think>가 있으면 마지막 </think>까지 0, 이후 본문(+종료 토큰)만 1. 없으면 assistant 전체가 1.",
    "- 손실 가중치: user/템플릿=0, <think>/</think>=1, think 내부=0.1, 최종 답변(+종료 토큰)=1.",
)

data_py.write_text(src, encoding="utf-8")

# model.py: binary masking CE -> token weighted CE
# train.py 호환성을 위해 변수명 loss_mask는 유지한다.
model_py = Path("model.py")
msrc = model_py.read_text(encoding="utf-8")
forward_start = msrc.find("    def forward(self, input_ids: torch.Tensor, loss_mask: torch.Tensor | None = None):")
forward_end = msrc.find("    @torch.inference_mode()", forward_start)
if forward_start < 0 or forward_end < 0:
    raise RuntimeError("model.py의 forward() 위치를 찾지 못했습니다.")

new_forward = '''    def forward(self, input_ids: torch.Tensor, loss_mask: torch.Tensor | None = None):
        # loss_mask는 token loss weight: 0=ignore, 0.1=think 내부, 1=full
        h = self._trunk(input_ids)
        if loss_mask is None:
            return self.logits_from_hidden(h)

        B, T = input_ids.shape

        def weighted_ce(logits: torch.Tensor, targets: torch.Tensor,
                        token_weights: torch.Tensor) -> torch.Tensor:
            token_weights = token_weights.float()
            targets = targets.masked_fill(token_weights <= 0, -100)
            per_token = F.cross_entropy(
                logits.reshape(-1, self.cfg.vocab_size).float(),
                targets.reshape(-1),
                ignore_index=-100,
                reduction="none",
            ).view_as(token_weights)
            denom = token_weights.sum().clamp_min(1e-8)
            return (per_token * token_weights).sum() / denom

        # main: target token 자신의 weight 사용
        main_logits = self.logits_from_hidden(h[:, :-1])
        main_targets = input_ids[:, 1:]
        main_weights = loss_mask[:, 1:]
        main_loss = weighted_ce(main_logits, main_targets, main_weights)

        # MTP: 미래 target token에도 같은 weight 사용
        mtp_losses = []
        mtp_offset_weights = []
        for k in range(1, self.cfg.mtp_n + 1):
            if T <= k + 1:
                break

            target_weights = loss_mask[:, 1 + k:]
            if not (target_weights > 0).any():
                continue

            hk = self.mtp_head(h[:, : T - 1 - k], offset_idx=k - 1)
            logits_k = self.logits_from_hidden(hk)
            targets_k = input_ids[:, 1 + k:]
            loss_k = weighted_ce(logits_k, targets_k, target_weights)

            # 기존 MTP offset 감쇠는 유지
            offset_w = 1.0 - 0.5 * (k - 1) / max(self.cfg.mtp_n - 1, 1)
            mtp_losses.append(loss_k * offset_w)
            mtp_offset_weights.append(offset_w)

        mtp_loss = (
            torch.stack(mtp_losses).sum() / sum(mtp_offset_weights)
            if mtp_losses else main_loss.new_zeros(())
        )
        loss = main_loss + self.cfg.mtp_weight * mtp_loss
        return {"loss": loss, "main_loss": main_loss.detach(), "mtp_loss": mtp_loss.detach()}

'''

msrc = msrc[:forward_start] + new_forward + msrc[forward_end:]
model_py.write_text(msrc, encoding="utf-8")

compile(data_py.read_text(encoding="utf-8"), "data.py", "exec")
compile(model_py.read_text(encoding="utf-8"), "model.py", "exec")

patched_data = data_py.read_text(encoding="utf-8")
patched_model = model_py.read_text(encoding="utf-8")
assert 'THINK_INNER_WEIGHT = 0.1' in patched_data
assert '_thinkw010_v1' in patched_data
assert 'dtype=np.float16' in patched_data
assert 'astype(np.float32)' in patched_data
assert 'reduction="none"' in patched_model
assert '(per_token * token_weights).sum() / denom' in patched_model
assert 'target_weights = loss_mask[:, 1 + k:]' in patched_model

print("[OK] data.py + model.py weighted SFT 패치 완료")
print("prompt=0 / tags=1 / think-inner=0.1 / final=1")


In [ ]:
# 2) Drive 및 사전학습 산출물
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
PRETRAIN_DIR = Path('/content/drive/MyDrive/korean_sllm_data/pretrain')
INIT_FROM = PRETRAIN_DIR / 'best.pt'
CKPT_DIR = '/content/drive/MyDrive/korean_sllm_data/pair'
SFT_DATA_DIR = Path(CKPT_DIR)
RESUME = None  # SFT 재개: str(Path(CKPT_DIR) / 'last.pt')
Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
if RESUME:
    assert Path(RESUME).is_file(), f'SFT 재개 체크포인트가 없습니다: {RESUME}'
    tokenizer_path = Path(CKPT_DIR) / 'spm.model'
    assert tokenizer_path.is_file(), f'SFT 토크나이저가 필요합니다: {tokenizer_path}'
    shutil.copy2(tokenizer_path, 'tokenizer/spm.model')
else:
    assert INIT_FROM.is_file(), f'먼저 사전학습을 실행하세요: {INIT_FROM}'
    assert (PRETRAIN_DIR / 'spm.model').is_file(), '사전학습 토크나이저가 필요합니다.'
    assert not any(Path(CKPT_DIR).glob('*.pt')), '기존 실행이 있습니다. RESUME을 지정하거나 새 CKPT_DIR을 사용하세요.'
    shutil.copy2(PRETRAIN_DIR / 'spm.model', 'tokenizer/spm.model')
    shutil.copy2('tokenizer/spm.model', Path(CKPT_DIR) / 'spm.model')


In [ ]:
# 3) 인스트럭션 데이터 준비: Drive JSONL 우선, 없으면 저장소 tar.xz
from data import ensure_dataset
for split in ('train', 'val'):
    source = SFT_DATA_DIR / f'{split}.jsonl'
    if source.is_file():
        shutil.copy2(source, f'{split}.jsonl')
    else:
        ensure_dataset(Path.cwd(), split)

import json
from pathlib import Path

def inspect_think_patterns(path='train.jsonl', examples=5):
    stats = {
        'rows': 0,
        'assistant_nonempty': 0,
        'has_open': 0,
        'has_close': 0,
        'has_both': 0,
        'starts_with_open': 0,
        'open_only': 0,
        'close_only': 0,
        'multi_close': 0,
        'body_after_last_close': 0,
        'empty_after_last_close': 0,
        'json_error': 0,
    }
    samples = []

    with Path(path).open(encoding='utf-8') as f:
        for line in f:
            stats['rows'] += 1
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                stats['json_error'] += 1
                continue

            assistant = str(obj.get('assistant', '') or '')
            if not assistant.strip():
                continue
            stats['assistant_nonempty'] += 1

            n_open = assistant.count('<think>')
            n_close = assistant.count('</think>')
            has_open, has_close = n_open > 0, n_close > 0
            stats['has_open'] += int(has_open)
            stats['has_close'] += int(has_close)
            stats['has_both'] += int(has_open and has_close)
            stats['starts_with_open'] += int(assistant.lstrip().startswith('<think>'))
            stats['open_only'] += int(has_open and not has_close)
            stats['close_only'] += int(has_close and not has_open)
            stats['multi_close'] += int(n_close > 1)

            if has_close:
                cut = assistant.rfind('</think>') + len('</think>')
                body = assistant[cut:]
                stats['body_after_last_close'] += int(bool(body.strip()))
                stats['empty_after_last_close'] += int(not body.strip())

                if len(samples) < examples:
                    samples.append({
                        'user': str(obj.get('user', ''))[:160].replace('\n', ' '),
                        'n_open': n_open,
                        'n_close': n_close,
                        'around_last_close': assistant[max(0, cut-100):cut+240].replace('\n', '\\n'),
                        'body_head': body.lstrip()[:240].replace('\n', '\\n'),
                    })

    print(f"\n=== {path} <think> 통계 ===")
    for k, v in stats.items():
        pct = (f" ({v / stats['assistant_nonempty'] * 100:.2f}%)"
               if stats['assistant_nonempty'] and k not in ('rows', 'assistant_nonempty', 'json_error') else '')
        print(f"{k:26s}: {v:,}{pct}")

    print('\n=== 마지막 </think> 기준 샘플 ===')
    for i, s in enumerate(samples, 1):
        print(f"\n[{i}] open={s['n_open']} close={s['n_close']} | user: {s['user']}")
        print('around:', s['around_last_close'])
        print('body  :', s['body_head'])
    return stats

think_stats = inspect_think_patterns('train.jsonl')


In [ ]:
# 3b) <think> token weight 동작 확인
import importlib
from collections import Counter

import data
importlib.reload(data)
from data import load_tokenizer, encode_sample

sp_check = load_tokenizer()
test_answer = (
    "<think>중간 추론 1입니다.</think>\n"
    "<think>중간 추론 2입니다.</think>\n\n"
    "이 부분은 최종 답변이므로 weight 1로 학습되어야 합니다."
)
test_ids, test_weights = encode_sample(sp_check, "테스트 질문", test_answer)

answer_text = f"{test_answer}<end_of_turn>"
proto = sp_check.encode(answer_text, out_type="proto")
answer_len = len(proto.pieces) + 1
prompt_len = len(test_ids) - answer_len
piece_weights = test_weights[prompt_len:prompt_len + len(proto.pieces)]

print("=== weight별 token 수 ===")
for w, n in sorted(Counter(round(float(w), 3) for w in test_weights).items()):
    print(f"weight {w:>3}: {n:,} tokens")

assistant_bytes = test_answer.encode("utf-8")
open_tag = b"<think>"
close_tag = b"</think>"
search_pos = 0
pair_count = 0
interior_piece_count = 0

while True:
    open_start = assistant_bytes.find(open_tag, search_pos)
    if open_start < 0:
        break
    inner_start = open_start + len(open_tag)
    close_start = assistant_bytes.find(close_tag, inner_start)
    if close_start < 0:
        break
    close_end = close_start + len(close_tag)
    pair_count += 1

    for piece, w in zip(proto.pieces, piece_weights):
        if piece.begin >= inner_start and piece.end <= close_start:
            assert abs(float(w) - 0.1) < 1e-3
            interior_piece_count += 1

        overlaps_open = piece.begin < inner_start and piece.end > open_start
        overlaps_close = piece.begin < close_end and piece.end > close_start
        if overlaps_open or overlaps_close:
            assert abs(float(w) - 1.0) < 1e-6

    search_pos = close_end

assert pair_count == 2
assert interior_piece_count > 0

# 마지막 </think> 이후는 전부 full weight
last_close_end = assistant_bytes.rfind(close_tag) + len(close_tag)
final_piece_count = 0
for piece, w in zip(proto.pieces, piece_weights):
    if piece.begin >= last_close_end:
        assert abs(float(w) - 1.0) < 1e-6
        final_piece_count += 1
assert final_piece_count > 0

reason_ids = [
    piece.id for piece, w in zip(proto.pieces, piece_weights)
    if abs(float(w) - 0.1) < 1e-3
]
print("\n=== weight=0.1 reasoning token decoded ===")
print(sp_check.decode(reason_ids))

# 일반 assistant 응답은 prompt 제외 전부 1
plain_answer = "일반 답변입니다."
plain_ids, plain_weights = encode_sample(sp_check, "테스트 질문", plain_answer)
plain_proto = sp_check.encode(f"{plain_answer}<end_of_turn>", out_type="proto")
plain_prompt_len = len(plain_ids) - (len(plain_proto.pieces) + 1)

assert all(abs(float(w)) < 1e-8 for w in plain_weights[:plain_prompt_len])
assert all(abs(float(w) - 1.0) < 1e-6 for w in plain_weights[plain_prompt_len:])

print("\n[OK] prompt=0.0")
print("[OK] <think>/</think>=1.0")
print("[OK] think 내부=0.1")
print("[OK] 최종 답변/end_of_turn/EOS=1.0")
print("[OK] 일반 assistant 응답=1.0")


In [ ]:
# 4) 사전학습 -> SFT. 재개 시 2번 셀의 RESUME을 pair/last.pt 경로로 설정하세요.
import subprocess
# 새 실행마다 캐시를 분리해 변경한 SFT 데이터의 구 캐시 재사용을 방지합니다.
import tempfile
sft_cache = tempfile.mkdtemp(prefix='sft_think010_', dir='/content')
cmd = [
    'python', 'train.py', '--stage', 'sft',
    '--seq-len', '2048', '--batch-size', '8', '--grad-accum', '4',
    '--epochs', '2', '--warmup-steps', '100', '--lr', '5e-5',
    '--eval-every', '250', '--save-every', '500',
    '--cache-dir', sft_cache, '--ckpt-dir', CKPT_DIR,
]
cmd += ['--resume', str(RESUME)] if RESUME else ['--init-from', str(INIT_FROM)]
subprocess.run(cmd, check=True)


In [ ]:
# 5) 생성 데모 - val main_loss 최저 시점인 best.pt 로드. 답변 길이 p90 392 / p99 839 tokens (32k 실측) 이므로 512 이상으로 (256 은 중앙값 134 만 겨우 넘김)
import torch
from data import load_tokenizer, encode_sample
from model import KoreanSLLM, ModelConfig

ckpt = torch.load(f'{CKPT_DIR}/best.pt', map_location='cuda', weights_only=True)
model = KoreanSLLM(ModelConfig(**ckpt['config'])).cuda()
model.load_state_dict(ckpt['model'])
sp = load_tokenizer()

prompt = '감기에 걸렸을 때 어떻게 해야 하나요?'
ids = encode_sample(sp, prompt, '')[0][:-2]
out = model.generate(torch.tensor([ids], device='cuda'), max_new_tokens=512, temperature=0.7)
print(sp.decode(out[0].tolist()))

In [ ]:
# 5b) self-speculative decoding 데모 (batch=1) - MTP 헤드로 draft 를 뽑아 트렁크 1회 forward 로 병렬 검증
#     greedy(temperature=0)는 generate 와 출력 동일이 보장되고, sampling 은 rejection 으로 동일 분포 보존
#     (단, RNG 소비가 달라 아래 두 출력의 내용 자체는 다를 수 있음). draft_k 는 offset별 수용률을 보고 조정.
import time

def timed(fn):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = fn()
    torch.cuda.synchronize(); return out, time.perf_counter() - t0

inp = torch.tensor([ids], device='cuda')
base, t_base = timed(lambda: model.generate(inp, max_new_tokens=512, temperature=0.7))
(spec, stats), t_spec = timed(lambda: model.generate_speculative(
    inp, max_new_tokens=512, temperature=0.7, draft_k=8, return_stats=True))

n_base = base.shape[1] - inp.shape[1]
n_spec = spec.shape[1] - inp.shape[1]
acc, prop = sum(stats['accepted']), sum(stats['proposed'])
print(f'generate            : {n_base:4d} tokens, {t_base:.2f}s ({n_base / t_base:.1f} tok/s)')
print(f'generate_speculative: {n_spec:4d} tokens, {t_spec:.2f}s ({n_spec / t_spec:.1f} tok/s)')
print(f'draft 수용률 {acc}/{prop} ({acc / max(prop, 1):.0%}), offset별:',
      ' '.join(f'{k+1}:{a}/{p}' for k, (a, p) in enumerate(zip(stats['accepted'], stats['proposed'])) if p))
print()
print(sp.decode(spec[0].tolist()))

In [ ]:
# 6) Drive 에 쓴 체크포인트를 확실히 저장 - 버퍼 flush 후 언마운트 (이후 Drive 경로 접근 불가, 필요하면 2번 셀로 재마운트)
import os
print('저장된 체크포인트:', sorted(f for f in os.listdir(CKPT_DIR) if f.endswith('.pt')))
drive.flush_and_unmount()
print('Drive flush + unmount 완료')